# Datascape 2026: package-condition classification pipeline (Aman / Rusak / Lainnya)

Reproducible end-to-end notebook for the team's current best submission and the scenarios built on it. Metric: macro-F1; public LB = 30% of test.

**What is cached and what is recomputed.** Three things are loaded from cache, not recomputed from raw images here:
- the frozen-tower embeddings (`work/emb/*.npz`, extracted by `colab/emb_giant.py` / `kernels/emb*`);
- the SSCD copy-detector neighbours (`work/nntr_sscd.npz`, `work/sscd.py`);
- the group folds.

Pixel-copy scores are recomputed when `RECOMPUTE_PIX = True` (~90 s).

**Two modes**
- `RETRAIN = False` (default, under a minute): rebuilds every submitted file from the cached head predictions and **asserts row-for-row equality** with what was submitted. That assertion is the reproducibility proof.
- `RETRAIN = True` (or env `DS_RETRAIN=1`): retrains the domain-adapted MLP heads from the embeddings (2 tower sets × 5 seeds; ~10 min on Apple MPS, ~2 min on CUDA). It reports the agreement with the cached predictions and with the submitted file. Seeds differ across CPU/MPS/CUDA, so agreement is reported, not asserted.

**Sections**
1. Data audit: sources (identified by image size), class mix, noise strata
2. Duplicates and leakage: pixel-exact copies (dihedral NCC), copy structure per source
3. Validation design: group folds and their leak check
4. Model: frozen vision towers + MLP heads (group-OOF)
5. Label-noise audit with cleanlab
6. Twin lock + domain-adapted heads → `d04_da_bt2` (public LB 0.84521)
7. Post-processing probes for 10-05 → `d05_p416_allA`, `d05_p675_R`
8. Iterative pseudo-labelling (test-domain held-out check)
9. Experiment ranking and expectation vs reality

AI-use declaration: code was written with an AI coding assistant (Claude); no VLM/LLM inference is used on the data. Vision towers are used as frozen image encoders only (no text, no zero-shot), as allowed by the organiser.

In [1]:
import os, json, hashlib, time, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, confusion_matrix
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 80)
ROOT = Path(os.environ.get('DS_ROOT', Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd())).resolve()
W, SUBS = ROOT / 'work', ROOT / 'submissions'
RETRAIN = os.environ.get('DS_RETRAIN') == '1'   # True: retrain domain-adapted heads from embeddings
RECOMPUTE_PIX = False    # True: recompute pixel-exact copy scores from the images (~90 s)
C = ['Aman', 'Lainnya', 'Rusak']; T0 = time.time()
print('root', ROOT)

root /Users/erwinqodariyah/Claude Project/Kaggle/Datascape


## 1. Data audit
Image size identifies the data source; sources differ wildly in Rusak rate and in their train/test mix.

In [2]:
meta = pd.read_csv(W / 'emb/meta_sorted.csv')            # one row per image, fixed order shared by every artifact
TR = (meta.split == 'train').values; TE = ~TR
BIG = ['224x224', '416x416', '512x512', '640x640', '675x900']
meta['size'] = meta.w.astype(str) + 'x' + meta.h.astype(str); meta['source'] = np.where(meta['size'].isin(BIG), meta['size'], 'other')
LAB = np.array([C.index(l) if isinstance(l, str) else -1 for l in meta.label])   # -1 for test rows
def strata(m):  # salt-and-pepper / heavy-noise flag x 224px source (noise was injected mostly into train Aman rows)
    nz = pd.read_csv(W / 'meta_noise.csv').set_index('path').loc[m.path]
    noisy = (nz.impulse.values > 0.02) | (nz.medabs.values >= 6)
    return np.where(noisy, 'noisy', 'clean') + '_' + np.where(((m.w == 224) & (m.h == 224)).values, '224', 'other')
ST = strata(meta)
src = pd.DataFrame({'train_n': meta[TR].source.value_counts(), 'train_rusak': meta[TR].groupby('source').label.apply(lambda s: (s == 'Rusak').mean()).round(3),
                    'test_n': meta[TE].source.value_counts()}).fillna(0)
src['train_share'] = (src.train_n / TR.sum()).round(3); src['test_share'] = (src.test_n / TE.sum()).round(3)
print('train', TR.sum(), 'test', TE.sum(), '| train class mix', meta[TR].label.value_counts(normalize=True).round(3).to_dict())
print(pd.crosstab(ST[TR], meta.label[TR], normalize='index').round(3))
src

train 12224 test 3057 | train class mix {'Aman': 0.72, 'Rusak': 0.246, 'Lainnya': 0.035}
label         Aman  Lainnya  Rusak
row_0                             
clean_224    1.000    0.000  0.000
clean_other  0.648    0.039  0.313
noisy_224    1.000    0.000  0.000
noisy_other  0.878    0.026  0.096


,train_n,train_rusak,test_n,train_share,test_share
source,,,,,
224x224,253,0.000,520,0.021,0.170
416x416,298,0.708,193,0.024,0.063
512x512,2492,0.082,45,0.204,0.015
640x640,2785,0.469,347,0.228,0.114
675x900,499,0.784,280,0.041,0.092
other,5897,0.151,1672,0.482,0.547


## 2. Duplicates and leakage
Pixel-exact matching, stricter than MD5: each image becomes a 128-px grayscale thumbnail, zero-mean and unit-norm. Its 8 dihedral variants (4 rotations × flip) are compared against its top-10 SSCD copy-detection candidates by normalised cross-correlation (NCC). NCC ≥ 0.99 counts as an exact copy, which survives re-encoding, resizing, rotation and flips.

In [3]:
from PIL import Image
D = np.load(W / 'nntr_sscd.npz'); I_sscd, S_sscd = D['i'], D['s']      # top-10 train neighbours by SSCD (copy detector), self excluded
def thumb(path, n=128):
    a = np.asarray(Image.open(path).convert('L').resize((n, n), Image.BILINEAR), np.float32); a -= a.mean(); return a / (np.linalg.norm(a) + 1e-6)
def pixel_ncc(paths, cand, n=128):
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex: TH = np.stack(list(ex.map(lambda p: thumb(p, n), paths)))
    DH = np.stack([np.rot90(b, k, axes=(1, 2)) for b in (TH, TH[:, :, ::-1]) for k in range(4)], 1)
    out = np.zeros(cand.shape, np.float32)
    for a in range(0, len(paths), 500):
        q = np.arange(a, min(a + 500, len(paths)))
        out[q] = np.einsum('qvd,qkd->qkv', DH[q].reshape(len(q), 8, -1), TH[cand[q]].reshape(len(q), cand.shape[1], -1)).max(2)
    return out
NCC = pixel_ncc((ROOT / meta.path).astype(str).values, I_sscd) if RECOMPUTE_PIX else np.load(W / 'pix_all.npz')['ncc']
best = NCC.max(1); exact = best >= 0.99
# train-train exact copies: how often do their labels agree? (upper bound on random label noise)
rows = [(meta.source[r], LAB[r], LAB[I_sscd[r, k]]) for r in np.where(TR)[0] for k in np.where(NCC[r] >= 0.99)[0]]
tt = pd.DataFrame(rows, columns=['source', 'y', 'y_copy']); tt['agree'] = tt.y == tt.y_copy
print('train-train exact pairs', len(tt), '| label agreement', round(tt.agree.mean(), 4)); print(tt.groupby('source').agree.agg(['size', 'mean']).round(3))
print('test rows with an exact train copy:', exact[TE].sum(), f'({exact[TE].mean():.1%})')

train-train exact pairs 6439 | label agreement 0.9621
         size   mean
source              
224x224    21  1.000
416x416     8  1.000
512x512   118  0.958
640x640  4180  0.958
675x900    40  1.000
other    2072  0.970
test rows with an exact train copy: 796 (26.0%)


**Copy structure per source.** In the synthetic and curated sources (416, 512, 675) only Aman rows have test copies; no train Rusak row does. The `d05_p675_R` probe is built on this.

In [4]:
te_idx = np.where(TE)[0]; hit = set(); copy_lab = {}
for r in te_idx:
    k = np.where(NCC[r] >= 0.99)[0]
    if len(k): hit |= set(I_sscd[r, k]); copy_lab[r] = pd.Series(LAB[I_sscd[r, k]]).mode()[0]
cs = []
for s in BIG + ['other']:
    T = te_idx[meta.source.values[te_idx] == s]; trs = np.where(TR & (meta.source == s).values)[0]; cl = [copy_lab[r] for r in T if r in copy_lab]
    cs.append(dict(source=s, test=len(T), test_exact_copy=len(cl), copy_aman=cl.count(0), copy_rusak=cl.count(2), train_rusak=(LAB[trs] == 2).sum(),
                   train_rusak_with_test_copy=sum(t in hit for t in trs if LAB[t] == 2), train_aman=(LAB[trs] == 0).sum(), train_aman_with_test_copy=sum(t in hit for t in trs if LAB[t] == 0)))
pd.DataFrame(cs)

,source,test,test_exact_copy,copy_aman,copy_rusak,train_rusak,train_rusak_with_test_copy,train_aman,train_aman_with_test_copy
0,224x224,520,53,53,0,0,0,253,28
1,416x416,193,21,21,0,211,0,87,11
2,512x512,45,5,5,0,205,0,2259,11
3,640x640,347,246,71,174,1305,238,1478,122
4,675x900,280,51,51,0,391,0,108,32
5,other,1672,420,361,59,889,68,4615,502


## 3. Validation design
Random K-fold rewards memorising near-duplicates: OOF 0.897 vs LB 0.786. Folds are therefore built on DINOv2 cosine ≥ 0.90 clusters (`folds_group90.csv`). Check: exact-copy train pairs should almost never straddle two folds.

In [5]:
fold_all = pd.read_csv(W / 'folds_group90.csv').set_index('path').fold
fold = fold_all.loc[meta.path[TR]].values; fpos = dict(zip(meta.path[TR], fold))
pairs = [(fpos[meta.path[r]], fpos[meta.path[I_sscd[r, k]]]) for r in np.where(TR)[0] for k in np.where(NCC[r] >= 0.99)[0]]
print('fold sizes', np.bincount(fold).tolist(), '| exact-copy pairs crossing folds:', f'{np.mean([a != b for a, b in pairs]):.2%}', 'of', len(pairs))

fold sizes [2445, 2445, 2445, 2445, 2444] | exact-copy pairs crossing folds: 6.71% of 6439


## 4. Model: frozen vision towers + MLP heads
Fine-tuning L-size backbones (CLIP-L, ConvNeXt, EVA02, PE-L, DINOv2-L, SigLIP2) always trailed frozen towers with an MLP head (§9).
- **Towers.** Seven frozen towers, each embedding median-filtered images under 5 geometric views (summed): PE-Core-L/14@336, SigLIP2-so400m@384, DINOv2-L, CLIP-L, DFN5B CLIP-H, EVA02-CLIP-L and PE-Core-G@448. A second set adds four more giant CLIP-family towers.
- **Head.** MLP 512 (dropout 0.5/0.3, label smoothing 0.1, AdamW + OneCycle, 40 epochs).
- **Ensemble.** `h2` = geometric mean of the two feature sets × 5 seeds.

In [6]:
def load_preds(tag):
    tr_paths = meta.path[TR].values; pos = {p: i for i, p in enumerate(tr_paths)}; tpos = {p: i for i, p in enumerate(meta.path[TE].values)}
    oof = np.full((TR.sum(), 3), np.nan); tst = np.zeros((TE.sum(), 3)); fs = sorted((W / 'preds' / tag).glob(f'pred_{tag}_f*.npz'))
    for f in fs:
        d = np.load(f, allow_pickle=True); oof[[pos[p] for p in d['val_path']]] = d['val']; tst[[tpos[p] for p in d['test_path']]] += d['test']
    assert len(fs) == 5 and not np.isnan(oof).any(); return oof, tst / len(fs)
y = LAB[TR]; oof_h2, _ = load_preds('h2_peg_c11')
print('h2 group-OOF macro-F1 vs original labels: %.4f' % f1_score(y, oof_h2.argmax(1), average='macro'), '| per class', f1_score(y, oof_h2.argmax(1), average=None).round(3))
pd.DataFrame(confusion_matrix(y, oof_h2.argmax(1)), index=[f'true {c}' for c in C], columns=[f'pred {c}' for c in C])

h2 group-OOF macro-F1 vs original labels: 0.8926 | per class [0.936 0.955 0.787]


,pred Aman,pred Lainnya,pred Rusak
true Aman,8544,8,248
true Lainnya,28,394,1
true Rusak,892,0,2109


## 5. Label-noise audit (cleanlab, Confident Learning)
Run on honest group-OOF probabilities. Relabelling the flagged rows lost 0.019 on LB earlier (§9), so the audit is used as a diagnostic, not as a relabelling step.

In [7]:
from cleanlab.filter import find_label_issues
from cleanlab.count import compute_confident_joint, estimate_latent
iss = find_label_issues(y, oof_h2, filter_by='both', n_jobs=1)
print('flagged', iss.sum(), 'of', len(y), '|', pd.crosstab(pd.Series([C[i] for i in y[iss]], name='given'), pd.Series([C[i] for i in oof_h2[iss].argmax(1)], name='model')).to_dict())
s_tr = meta.source.values[TR]; rows = []
for s in BIG + ['other']:
    m = s_tr == s
    if (np.bincount(y[m], minlength=3)[[0, 2]] > 5).all(): rows.append(dict(source=s, n=m.sum(), given_rusak=round(np.mean(y[m] == 2), 3), latent_rusak=round(estimate_latent(compute_confident_joint(y[m], oof_h2[m]), y[m])[0][2], 3)))
pd.DataFrame(rows)

flagged 437 of 12224 | {'Aman': {'Aman': 0, 'Lainnya': 14, 'Rusak': 272}, 'Lainnya': {'Aman': 7, 'Lainnya': 0, 'Rusak': 0}, 'Rusak': {'Aman': 144, 'Lainnya': 0, 'Rusak': 0}}


,source,n,given_rusak,latent_rusak
0,416x416,298,0.708,0.668
1,512x512,2492,0.082,0.085
2,640x640,2785,0.469,0.305
3,675x900,499,0.784,0.770
4,other,5897,0.151,0.144


## 6. Twin lock + domain-adapted heads → `d04_da_bt2` (public LB 0.84521)
1. **Twin vote.** Each test row takes a similarity-weighted vote of its SSCD train twins (sim ≥ 0.55). A row is locked to the vote when purity ≥ 0.9 and it is in the clean, non-224 stratum (train leave-one-out accuracy 0.981).
2. **Domain-adapted (DA) heads.** The heads are retrained on train plus the locked test rows, so they see the test-domain shift.
3. **Unlocked rows.**
   - No-twin rows: Rusak logit bias −0.8 (cross-validated on held-out locked rows).
   - Mixed-vote rows: a Rusak share calibrated from train vote bins.
   - Other strata: train P(Rusak | stratum).

In [8]:
def vote(I, S, thr):
    w = np.where(S >= thr, S, 0.0); P = np.stack([(w * (LAB[I] == c)).sum(1) for c in range(3)], 1); tot = P.sum(1)
    return np.where(tot > 0, P.argmax(1), -1), np.where(tot > 0, P.max(1) / np.maximum(tot, 1e-9), 0)
def match(L, st, ytr, sttr, targets, min_n=50):   # per-stratum Rusak logit shift to hit a target share (Saerens 2002-style quantification)
    L = L.copy()
    for s in np.unique(st):
        m = st == s; tgt = targets.get(s, (ytr[sttr == s] == 2).mean() if (sttr == s).any() else None)
        if tgt is None or m.sum() < min_n or tgt == 0: continue
        lo, hi = -8.0, 8.0
        for _ in range(60):
            b = (lo + hi) / 2
            if ((L[m] + np.r_[0, 0, b]).argmax(1) == 2).mean() < tgt: lo = b
            else: hi = b
        L[m, 2] += (lo + hi) / 2
    return L
def build_da_bt2(test_probs, thr=0.55, purity=0.9, bstar=-0.8):
    te = np.where(TE)[0]; lab, pur = vote(I_sscd[te], S_sscd[te], thr); lock = (lab >= 0) & (pur >= purity) & (ST[te] == 'clean_other')
    L = np.log(test_probs + 1e-9); ytr, sttr = LAB[TR], ST[TR]; final = L.argmax(1).copy(); tg = {}
    u = ~lock & (ST[te] == 'clean_other'); bt_share = float(((L[u] + np.r_[0, 0, bstar]).argmax(1) == 2).mean())
    for s in np.unique(ST[te]):
        m = ST[te] == s; T = {'clean_other': 0.381}.get(s, (ytr[sttr == s] == 2).mean()); n_l = (m & lock).sum()
        tg[s] = float(np.clip((T * m.sum() - (lab[m & lock] == 2).sum()) / max((m & ~lock).sum(), 1), 0, 1)) if n_l else T
        if s == 'clean_other': tg[s] = bt_share
    stu = ST[te].astype(object).copy(); Itr, Str = I_sscd[TR], S_sscd[TR]; lt, pt = vote(Itr, Str, thr)
    def wr(I_, S_): w = np.where(S_ >= thr, S_, 0); return (w * (LAB[I_] == 2)).sum(1) / np.maximum(w.sum(1), 1e-9)
    mtr = (lt >= 0) & (pt < 0.9) & (ST[TR] == 'clean_other'); rtr = wr(Itr, Str); edges = [0, 0.3, 0.5, 0.7, 1.01]
    cal = [float((LAB[TR][mtr & (rtr >= a) & (rtr < b)] == 2).mean()) for a, b in zip(edges[:-1], edges[1:])]
    mix = (lab >= 0) & ~lock & (ST[te] == 'clean_other'); nt = (lab < 0) & (ST[te] == 'clean_other'); rte = wr(I_sscd[te], S_sscd[te])
    stu[mix] = 'co_mixed'; stu[nt] = 'co_notwin'
    tg['co_mixed'] = float(np.mean([cal[np.searchsorted(edges, r, 'right') - 1] for r in rte[mix]])); tg['co_notwin'] = float(((L[nt] + np.r_[0, 0, bstar]).argmax(1) == 2).mean())
    final[~lock] = match(L[~lock], stu[~lock].astype(str), ytr, sttr, tg).argmax(1); final[lock] = lab[lock]
    print(f'locked {lock.sum()} | targets', {k: round(v, 3) for k, v in tg.items()}); return final, lock
def to_sub(pred):
    ss = pd.read_csv(ROOT / 'datascape/sample_submission.csv'); stem = {f.rsplit('.', 1)[0]: f for f in ss.filename}
    sub = pd.DataFrame({'filename': [stem[p.split('/')[-1][:-4]] for p in meta.path[TE].values], 'label': [C[i] for i in pred]})
    return ss[['filename']].merge(sub, on='filename', how='left')
_, t_da = load_preds('da_h2')
if RETRAIN:  # same recipe as work/da_heads.py: MLP heads on train + twin-locked test rows, geometric mean of 2 tower sets x 5 seeds
    import torch, torch.nn as nn, torch.nn.functional as F
    dev = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'; EP = int(os.environ.get('DS_EP', 40))
    CAT6 = ['vit_pe_core_large_patch14_336', 'vit_so400m_patch16_siglip_384', 'vit_large_patch14_dinov2', 'vit_large_patch14_clip_224', 'vit_huge_patch14_clip_224', 'eva02_large_patch14_clip_224']
    GIANT = ['vit_pe_core_gigantic_patch14_448', 'eva02_enormous_patch14_clip_224', 'vit_huge_patch14_clip_378', 'vit_giantopt_patch16_siglip_384', 'vit_gigantic_patch14_clip_378']
    def feat(t):
        E = np.load(W / f'emb/emb_{t}.npz'); X = sum(E[k].astype(np.float32) for k in E.files); return X / np.linalg.norm(X, axis=1, keepdims=True)
    def fit(X, idx, yy, seed, ep=EP):
        torch.manual_seed(seed); Xa = X[idx]
        m = nn.Sequential(nn.Dropout(0.5), nn.Linear(X.shape[1], 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, 3)).to(dev)
        opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-2); sch = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=ep * ((len(yy) + 255) // 256))
        for _ in range(ep):
            for b in torch.randperm(len(yy), device=dev).split(256):
                loss = F.cross_entropy(m(Xa[b]), yy[b], label_smoothing=0.1); opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        return m.eval()
    te = np.where(TE)[0]; lab_t, pur_t = vote(I_sscd[te], S_sscd[te], 0.55); lk = (lab_t >= 0) & (pur_t >= 0.9) & (ST[te] == 'clean_other')
    idx = np.r_[np.where(TR)[0], te[lk]]; yy = torch.tensor(np.r_[LAB[TR], lab_t[lk]], device=dev); Ls = []
    for towers in (CAT6 + GIANT[:1], CAT6 + GIANT):
        X = torch.tensor(np.concatenate([feat(t) for t in towers], 1), device=dev); P = 0
        for s in range(5):
            m = fit(X, idx, yy, s)
            with torch.no_grad(): P = P + m(X[te]).softmax(1).cpu().numpy() / 5
        Ls.append(np.log(P + 1e-9))
    t_new = np.exp(np.mean(Ls, 0)); t_new /= t_new.sum(1, keepdims=True)
    print(f'RETRAIN ({dev}, {EP} ep): argmax agreement with cached da_h2 {np.mean(t_new.argmax(1) == t_da.argmax(1)):.4f}'); t_da = t_new
final, lock = build_da_bt2(t_da); sub_da = to_sub(final)
ref = pd.read_csv(SUBS / 'd04_da_bt2.csv')
if RETRAIN: print(f'RETRAIN: agreement with submitted d04_da_bt2 {np.mean(sub_da.label.values == ref.label.values):.4f} (seed/device noise expected)')
else:
    assert (sub_da.filename == ref.filename).all() and (sub_da.label == ref.label).all(), 'd04_da_bt2 not reproduced'
    print('d04_da_bt2 reproduced row-for-row ✔ | mix', sub_da.label.value_counts(normalize=True).round(3).to_dict())

locked 1534 | targets {np.str_('clean_224'): np.float64(0.0), np.str_('clean_other'): 0.421, np.str_('noisy_224'): np.float64(0.0), np.str_('noisy_other'): np.float64(0.096), 'co_mixed': 0.555, 'co_notwin': 0.299}
d04_da_bt2 reproduced row-for-row ✔ | mix {'Aman': 0.708, 'Rusak': 0.274, 'Lainnya': 0.018}


## 7. Post-processing probes for 10-05
Each probe changes one source only, so its LB delta reads that source's true Rusak rate. `probe_curve` turns a delta into q, the fraction of flipped rows that are truly Rusak. It enumerates the public-LB confusion states consistent with 0.84521 (F1 Lainnya held at 0.96).
- `d05_p416_allA`: every Rusak prediction in the 416×416 synthetic source → Aman.
- `d05_p675_R`: 675×900 VN-parcel rows with no train copy (NCC ≤ 0.8) → Rusak. No train Rusak row in this source has a test copy, while every copied test row is Aman.

In [9]:
base = sub_da.set_index('filename').label; fn_meta = meta[TE].set_index('filename')
p416 = base.copy(); m416 = (fn_meta.loc[base.index, 'source'] == '416x416').values & (base.values == 'Rusak'); p416[m416] = 'Aman'
p675 = base.copy(); nc = pd.Series(best[TE], index=meta.filename[TE]).loc[base.index].values
m675 = (fn_meta.loc[base.index, 'source'] == '675x900').values & (nc <= 0.8) & (base.values == 'Aman'); p675[m675] = 'Rusak'
for name, p in [('d05_p416_allA', p416), ('d05_p675_R', p675)]:
    ref = pd.read_csv(SUBS / f'{name}.csv').set_index('filename').label.loc[base.index]
    if not RETRAIN: assert (ref == p).all(), name
    print(f'{name}: {"reproduced ✔" if (ref == p).all() else f"agreement {(ref == p).mean():.4f}"} | rows changed {(p != base).sum()} | Rusak share {(p == "Rusak").mean():.3f}')
def f1(tp, fp, fn): return 2 * tp / max(2 * tp + fp + fn, 1e-9)
def probe_curve(base, probe, S0, fL=0.96, f=0.3, tol=0.004):
    ch = base != probe; up = (base[ch] == 'Aman').all(); pA, pR = [round(f * (base == c).sum()) for c in ('Aman', 'Rusak')]; k = round(f * ch.sum())
    st = []
    for nR in range(int(0.4 * pR), int(1.6 * pR)):
        for tpR in range(max(0, nR - pA), min(nR, pR) + 1):
            fpR, fnR = pR - tpR, nR - tpR
            if pA - fnR >= 0 and abs((f1(pA - fnR, fnR, fpR) + fL + f1(tpR, fpR, fnR)) / 3 - S0) < tol: st.append((nR, tpR))
    out = {}
    for q in np.linspace(0, 1, 6):
        r, v = round(q * k), []
        for nR, tpR in st:
            fpR, fnR = pR - tpR, nR - tpR; tpA = pA - fnR
            if up: re = min(r, fnR); ae = min(k - re, tpA); a = (tpR + re, fpR + ae, fnR - re, tpA - ae, fnR - re)
            else: re = min(r, tpR); ae = min(k - re, fpR); a = (tpR - re, fpR - ae, fnR + re, tpA + ae, fnR + re)
            v.append((f1(a[3], a[4], a[1]) + fL + f1(a[0], a[1], a[2])) / 3)
        out[f'q={q:.1f}'] = round(float(np.mean(v)), 4)
    return out
pd.DataFrame({n: probe_curve(base, p, 0.84521) for n, p in [('d05_p416_allA', p416), ('d05_p675_R', p675)]}).T

d05_p416_allA: reproduced ✔ | rows changed 136 | Rusak share 0.230
d05_p675_R: reproduced ✔ | rows changed 158 | Rusak share 0.326


,q=0.0,q=0.2,q=0.4,q=0.6,q=0.8,q=1.0
d05_p416_allA,0.8747,0.8605,0.8458,0.8288,0.8133,0.7978
d05_p675_R,0.8113,0.8265,0.8425,0.8560,0.8700,0.8818


**Held for 10-06: exact-copy override.** These are test predictions that contradict a pixel-exact train copy. Exact train-train copies agree on labels 96.8% of the time.

In [10]:
te_pos = {r: i for i, r in enumerate(np.where(TE)[0])}; rows = []
for r, cl in copy_lab.items():
    pred = final[te_pos[r]]
    if pred != cl: rows.append(dict(filename=meta.filename[r], source=meta.source[r], copy_label=C[cl], da_bt2=C[pred], ncc=round(float(best[r]), 3)))
ex = pd.DataFrame(rows); print(len(ex), 'disagreements'); ex.groupby(['source', 'copy_label', 'da_bt2']).size()

20 disagreements


source   copy_label  da_bt2
416x416  Aman        Rusak     9
640x640  Aman        Rusak     5
         Lainnya     Aman      1
         Rusak       Aman      2
other    Rusak       Aman      3
dtype: int64

## 8. Iterative pseudo-labelling (test-domain held-out check)
- **Rounds.** Round r adds unlocked test rows whose round r−1 probability is ≥ 0.9 as pseudo-labels (`work/pl_iter.py`).
- **Evaluation.** Scored on twin-locked test rows. These are split into 5 folds by SSCD-twin connected components, so no held-out row has a near-copy in training.
- **Bar.** At least +0.005 F1 over round 0, with no rise in the unlocked Rusak share.

In [11]:
pl = W / 'pl_iter_scores.csv'
pd.read_csv(pl) if pl.exists() else 'pl_iter.py has not finished yet'

,round,heldout_f1,heldout_f1_b08,unlocked_rusak_b08,pseudo_rows
0,0,0.9604,0.9814,0.262,0
1,1,0.9613,0.9809,0.262,749
2,2,0.9628,0.9824,0.263,806
3,3,0.9613,0.9824,0.263,829


## 9. Experiment ranking and expectation vs reality
Ranked by strength of evidence: paired LB reads first, then single LB, then honest group-OOF against the original labels. Cleaned-label and leaky fine-tune OOF never set a rank, because that trap cost −0.085 once.

In [12]:
EVID = {'LB paired': 0, 'OOF honest + LB': 0, 'LB probe (pending 10-05)': 1, 'LB single': 2, 'test-domain held-out': 3, 'test-domain': 3, 'OOF honest': 4, 'OOF vs LB': 4,
        'OOF honest (cleaned)': 5, 'OOF leaky': 6, 'audit': 7, 'count': 7, 'design': 8}
ex = pd.read_csv(W / 'experiments.csv').fillna('')
num = lambda s: pd.to_numeric(pd.Series(s).astype(str).str.extract(r'(-?\+?\d*\.\d+)')[0], errors='coerce')
ex['_e'] = ex.evidence.map(EVID).fillna(9); ex['_lb'] = num(ex.lb_public).values; ex['_oof'] = num(ex.metric_oof_orig).values
ex = ex.sort_values(['_e', '_lb', '_oof'], ascending=[True, False, False], na_position='last').drop(columns=['_e', '_lb', '_oof']).reset_index(drop=True); ex.index += 1
ex[['category', 'experiment', 'metric_oof_orig', 'test_domain_metric', 'lb_public', 'expected_lb', 'evidence', 'decision']]

,category,experiment,metric_oof_orig,test_domain_metric,lb_public,expected_lb,evidence,decision
1,leakage+DA,SSCD twin lock (0.55 purity>=0.9 clean_other) + domain-adapted MLP heads (d0...,0.9318*,locked-row F1 0.978,0.84521,0.93,LB paired,ADOPTED (team best)
2,architecture,Plain h2 heads (MLP cat6+PE-G & cat11) at 27.4% Rusak (d05_h2_nolock_s337),0.8967,,0.83004,0.845,LB paired,BASELINE
3,architecture,MLP heads h2 = mean(cat6+PE-G & cat11) 5 seeds,0.8926,twin-locked 0.9299,0.83004,,OOF honest + LB,ADOPTED
4,cleaning,Confident-learning relabel 700 rows (r1003_clean_v1),0.7856,,0.76761,0.786,LB paired,CLOSED
5,postproc,416x416 synthetic source: all 136 Rusak preds -> Aman (d05_p416_allA),0.9318*,,,0.85,LB probe (pending 10-05),SUBMIT 10-05
6,leakage,675x900 VN parcels without train copy (158 rows) Aman -> Rusak (d05_p675_R),0.9318*,breakeven q~0.45,,0.855,LB probe (pending 10-05),SUBMIT 10-05
7,architecture,Team SAM-SLCA + ConvNeXt-S 384 (teammate),,,0.82086,,LB single,SUPERSEDED
8,leakage,Team super-ensemble + DINOv2 kNN twin lock,,,0.79,,LB single,CLOSED->REOPENED
9,architecture,0.5*5NN(DINOv2-B)+0.5*LR(DINOv2-L+CLIP-L) random folds,0.7987,,0.78627,0.799,LB single,SUPERSEDED
10,cleaning,cleanlab per-source prune (640x640: 449 rows) + DA retrain,,0 of 95 test rows change,,,test-domain,CLOSED


In [13]:
led = pd.read_csv(SUBS / 'ledger.csv')
led[['name', 'oof_macro_f1', 'expected_lb', 'exp_lo', 'exp_hi', 'actual_lb', 'delta_vs_expected']].dropna(subset=['expected_lb'])

,name,oof_macro_f1,expected_lb,exp_lo,exp_hi,actual_lb,delta_vs_expected
0,r1003_calib_knn_probe,0.7987,0.799,0.783,0.815,0.78627,-0.01273
1,r1003_clean_v1,0.7856,0.786,0.770,0.802,0.76761,-0.01839
2,d04_calib_priormatch,0.8462,0.831,0.811,0.851,NaN,NaN
3,d04_clipft_cat4_pm,0.8784,0.863,0.843,0.883,NaN,NaN
4,d04_clipft_cat6_pm,0.8812,0.866,0.846,0.886,NaN,NaN
5,d04_mlp6_pm,0.8894,0.874,0.854,0.894,NaN,NaN
6,d04_mlp6_pm38,0.8776,0.858,0.830,0.890,NaN,NaN
7,d04_sscd55_h2,0.9290,0.905,0.870,0.940,NaN,NaN
8,d04_sscd55p90_h2,0.9318,0.912,0.875,0.945,NaN,NaN
9,d04_sscd55p90_h2_bt2,0.9318,0.920,0.880,0.955,NaN,NaN


In [14]:
out = ROOT / 'notebook' / 'output'; out.mkdir(parents=True, exist_ok=True)
sub_da.to_csv(out / 'submission_d04_da_bt2.csv', index=False); p416.reset_index().to_csv(out / 'submission_d05_p416_allA.csv', index=False); p675.reset_index().to_csv(out / 'submission_d05_p675_R.csv', index=False)
for f in sorted(out.glob('*.csv')): print(f.name, hashlib.md5(f.read_bytes()).hexdigest()[:10])
print(f'notebook finished in {time.time() - T0:.0f}s')

submission_d04_da_bt2.csv 81fa36194d
submission_d05_p416_allA.csv 7b4d91044c
submission_d05_p675_R.csv 8a80b58e6f
notebook finished in 2s
